# M3E1 - Results Analysis on Arabidopsis Data with DESeq2 

In this Notebook, we will be analyzing the results of our pseudo-alignment using `salmon` on the *A. thaliana* OPT3 vs Col-0 dataset. At this point, you should have completed the labs and practices. We are going to be using `R` within this notebook within our `Module 3 - R` Kernel. If you have not completed this or had issues in M3P2, you can use the global environment `the course-wide Module 3 environment (see `environment/module_3_rnaseq_salmon.yml`)` and install `irkernel` to your Jupyter Session manually. 

This may be your first time seeing any coding in `R`, and it is alright if you do not understand the syntax. Most of the code is commented, and you can ask AI to explain the code line by line to gather a better understanding. What is important to take away from the Exercise will be the post-processing of RNA-Seq data, such as the tx2gene mapping, creating a list of differentially expressed genes from our list of counts, and understanding how to interpret the statistics behind our differential expression analysis.

To complete this Notebook, you will need to: 
- Complete the differential expression analysis on our data set, filling in code where it is needed.
- Answer ALL questions within the exercise worksheet.
- Use these results to answer the discussion post for this week.
               

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

## Steps in completing a differential analysis from `salomn`'s `quant.sf` count files on our *A. thaliana* OPT3 vs Col-0 dataset

### Overview of our dataset and pipeline so far: 

So far, we have completed a pseudo-alignment on the 6 *A. thaliana* samples. These 6 samples came from the [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275) series. We have two treatment conditions, with three replicates each: **1. Col-0 untreated roots** and **2. OPT3 untreated roots**. 
| Sample ID | GEO Accession | Condition | Replicate | Description |
|------------|----------------|------------|------------|--------------|
| **CRN1-1** | GSM2090459 | C (Col-0 untreated) | N-1 | Col-0 untreated root sample |
| **CRN2-1** | GSM2090462 | C (Col-0 untreated) | N-2 | Col-0 untreated root sample |
| **CRN3-1** | GSM2090465 | C (Col-0 untreated) | N-3 | Col-0 untreated root sample |
| **ORN1-1** | GSM2090477 | O (OPT3-untreated)| N-1 | OPT3 untreated root sample |
| **ORN2-1** | GSM2090480 | O (OPT3-untreated) | N-2 | OPT3 untreated root sample |
| **ORN3-1** | GSM2090483 | O (OPT3-untreated) | N-3 | OPT3 untreated root sample |

**REMINDER**: We know that the OPT3 mutant should express an iron deficiency response within the roots because a loss-of-function mutation in OPT3 in *Arabidopsis thaliana* leads to a constitutive iron deficiency response in the roots, even when sufficient iron is available. The OPT3 protein is crucial for transporting iron from the shoot to the roots, allowing the roots to "sense" the plant's overall iron sufficiency. Without functional OPT3, the root "thinks" the plant is deficient in iron and activates iron uptake pathways. **Therefore, to demonstrate the pipeline, we will investigate the difference in gene expression between the OPT3 mutant root treatment samples and our Col-0 root control samples.**

This was discussed in the paper covered in Module 2, [Cadmium interference with iron sensing reveals transcriptional programs sensitive and insensitive to reactive oxygen species](https://academic.oup.com/jxb/article/73/1/324/6367699). 


To get to our DE analysis, we have done the following: 
- Downloaded the raw reads (`fastq.gz` files) from the SRA database for each of our samples - M3L2
- Ran `fastp` to trim and check the quality of our reads for each of the 6 samples
- Used `salmon` to align the trimmed reads to our TAIR10 transcriptome to produce our counts (`quant.sf` and `quant.genes.sf`) for the 6 samples

Now, use `R` to summarize our counts to the gene level using a `tx2gene` table. We can use DE analysis programs, such as `DESeq2`, to identify genes whose expression level changes significantly across our treatment groups (OPT3 mutant roots vs Col-0 roots). By comparing our counts normalized with log2FC, or VST, across these conditions, we can detect genes that are up- or down-regulated by the treatment. **In the case of our dataset, we expect to see an iron deficiency response caused within the roots as a result of impaired sensing of iron due to the OPT3 mutation.**

### Inputs for DE analysis with `DESeq2`

We will use the files generated by `salmon` containing the counts of our transcripts identified in pseudo-alignment within TAIR10. These files, `quant.sf` contain columns of the following: 
- Name (transcript ID)
- Length (transcript length)
- EffectiveLength
- TPM
- NumReads (estimated raw counts)

We will import these `quant.sf` files to `R`, and then convert the transcript counts to the gene level. 

*NOTE:* We did produce gene-level counts from our salmon output in the form of `quant.genes.sf` files. In the exercise, we will still use a tx2gene table to demonstrate the conversion of these files, as there can be issues with the use of our gene-level counts produced by salmon, and because other pseudo-aligners generally do not produce gene-level counts (`kallisto`). 

### **IMPORTANT NOTE ON THE USE OF PROGRAMS SUCH AS `DESeq2` FOR USE IN QUANITIFICATION OF COUNTS FROM PSEUDO-ALIGNERS**

Pseudo-aligners, like `salmon` and `kallisto`, report transcript-level abundances: TPM, estimated counts, and effective lengths. These estimated counts are fractional numbers because the reads are assigned probabilistically to the transcripts using k-mers. These counts are **not** normalized for library size or sequencing depth, and they are raw counts in the sense that they are reads mapping to the likelihood of their assigned transcripts. 

**These estimated transcript level counts are assigned to genes to produce gene level counts, but because these counts are probability-based based they are fractional (assigned by likelihood). That means that when we are importing counts from our pseudo-aligners to a DE analysis software, like `DESeq2` or `EdgeR`, we are truncating the estimates of these counts into integers rather than floating-point (fractional) values.** These counts are still raw counts in the sense that they have not been adjusted for library size, but they are, in fact, truncated. 

This can cause issues when investigating genes that exhibit low expression, because the loss of their fractional assignment can affect the p-values of genes with few counts. In genes with high expression, the loss of the fractions within assignment is minimally impactful on the result in the case that p-values are calculated with counts in the 100s or 1000s. 

When using `DESeq2` from a `salmon` input, we will input our gene-level counts using tximport and let the `DESeq2` software work out the length normalization for our estimated "raw counts" from our `tx2gene` tables. This is sub-optimal if we were wanting to investigate genes that are infrequently expressed and could lead to bias within our experiment. For the sake of the data that we are working with, we expect to see heightened expression of genes like IRT1, FRO2, and other genes common in iron-deficiency response. This means the impact of this integer-based normalization within `DESeq2` will be minimal when looking at our top ~100 differentially expressed genes, assuming their counts are high to begin with. 

EX: If you sum fractional counts for a gene, the total may be something like 12.8 reads. Rounding or truncating to 12 reads is trivial for modeling, because `DESeq2` assumes negative binomial variance, which already accounts for count noise. Only genes with counts near zero could be affected, but DESeq2 handles low counts by shrinking log fold changes and down-weighting low-abundance genes.

### Loading libraries, counts data, and creating the TX2Gene tables.

To use libraries within R, you must import them. The following libraries will be imported to our `R` kernel:

In [ ]:
#load libraries (don't mind the conflicts)
library(tximport)         # Import Salmon quantifications
library(readr)            # Read CSV files
library(DESeq2)           # Differential expression
library(tidyverse)        # Data wrangling + ggplot2
library(pheatmap)         # Heatmaps
library(cowplot)          # Multi-plot layout
library(ComplexHeatmap)   # Advanced heatmaps
library(UpSetR)           # UpSet plots for overlapping DEGs
library(ggrepel)          # Improved labels for plots
library(clusterProfiler)  # Gene set enrichment
library(biomaRt)          # Source for Gene-Ontology Data
library(rtracklayer)

## Importing our samples and their counts into `R`

We need to import our sample's `quant.sf` files into `R`. These counts are stored within our folder `m3_project_pseudo_alignment/results/aligned_counts/{$SAMPLENAME}/quant.sf`. We also need to create a dataframe for each of our samples, matching their treatment to the sample. We will do this below: 

In [ ]:
#build sample metadata
colData <- data.frame(
    sample = c("CRN1-1", "CRN2-1", "CRN3-1", "ORN1-1", "ORN2-1", "ORN3-1"),
    condition = c("Col0","Col0","Col0","OPT3","OPT3","OPT3")
)

colData$condition <- factor(colData$condition, levels = c("Col0", "OPT3"))

colData #look at our sample treatment conditions

In [ ]:
#Import counts(quant.sf file paths)
#Import quant.sf transcript counts file paths from our 6 samples
quant_files <- file.path("../m3_project_pseudo_alignment/results/aligned_counts",
                         colData$sample, "quant.sf")

names(quant_files) <- colData$sample #label each file path with the sample name

head(quant_files) #check the locations of our quant.sf files

In [ ]:
#inspect our quant.sf files
q_reads <- read_tsv(quant_files)
head(q_reads) #check quant.sf (transcript counts) files format

### Build tx2gene file
We now need to build a `tx2gene` file, so that we can convert our transcript counts from the `quant.sf` file to gene-level counts for use in DE analysis using `DESeq2`. The `tx2gene` file, or in our case a data frame, is a list of transcripts (mRNA, tRNA, ncRNA, rRNA, etc) indexed to gene IDs. Typically a `tx2gene` table will look like this: 
![image.png](images/M3E1_RNA_Seq_Analysis_dc9dbbde.png)

- TXNAME - transcript ID from your reference (e.g., Salmon quantification).
- GENEID - gene ID that groups multiple transcripts (isoforms) together.

We use this file because `salmon` and other pseudo-aligners quantify transcripts, not genes. In our results, we want to see gene-level expression not the multiple isoforms that RNA-Seq reads can map to. We use a program called `tximport` to aggregate counts by genes from `salmon`'s `quant.sf` output and create a gene-level count matrix for use in `DESeq2`.  

In [ ]:
#Import the TAIR10 GFF to have Transcripts and Genes columns
gff <- import("../m3_project_pseudo_alignment/references/TAIR10_GFF3_genes.gff")
head(gff) ##check the formatting of our TAIR10 GFF

We are selecting for transcript types "mRNA", "tRNA", "ncRNA", "rRNA" within our `gff` dataframe from the `type` column in the TAIR10 gff file. 

In [ ]:
#select the transcript and geneid columns from our gff file
transcripts <- gff[gff$type %in% c("mRNA", "tRNA", "ncRNA", "rRNA")] #Transcripts are mRNA within the Type column of the TAIR10 gff file

tx2gene <- data.frame(
  TXNAME = transcripts$ID,      # transcript ID (mRNA ID)
  GENEID = transcripts$Parent   # gene ID (in Parent column, links transcripts to genes within the gff)
)

head(tx2gene) #Lets check our work

In [ ]:
#At this point, we need to mutate our tx2gene file so that the only two columns are the TXNAME and GENEID. 
tx2gene <- tx2gene %>%
  mutate(
    #TXNAME = sub("\\..*$", "", as.character(TXNAME)), #drop the .XX
    GENEID = GENEID.value #drop the .XX
  ) %>%
  dplyr::select(TXNAME, GENEID)# %>%   # explicitly use dplyr::select, masked by other to select only these two columns
  #dplyr::distinct() #Only include distinct entries to the table

head(tx2gene) #check tx2gene file

### Building txi for `DESeq2` input

We need to make a list of our samples' count matrices so that we can pipe them to DESeq2 to perform differential expression analysis. This list is called `txi` and we will use `tximport` to create it. `txi` will contain the following matrices: 

`txi$counts` → gene-level counts (integer-like). This is what DESeq2 uses for differential expression.

`txi$abundance` → TPM-level abundances. Useful for plotting or comparing differential expression across samples.

`txi$length` → effective transcript lengths. Used for length scaling TPM values.

Each of these matrices contains rows labeled as the `GENEID`'s and columns for the 6 `samples`. 

[tximport Documentation](https://bioconductor.org/packages/devel/bioc/vignettes/tximport/inst/doc/tximport.html)

In [ ]:
#build txi for import to dds(DESeqDataSet)
txi <- tximport(files = quant_files, #use the quant.sf files from each of our samples above - these are file paths
                type = "salmon", #tximport uses salmon input
                tx2gene = tx2gene, #our tx2gene file's called tx2gene
                #ignoreTxVersion = TRUE, #optional, will ignore versions (isoforms) of genes
                #countsFromAbundance = "no" # uses estimated raw counts from salmon
               ) 
head(txi$length)          # matrices available: counts, abundance, length
head(txi$counts)    # gene-level counts ready for DESeq2
head(txi$abundance) # TPM-level abundances

### Tximport to DDS for `DESeq2`

We will now convert our `txi` dataframe into DDS (DESeqDataSet) format for use in processing with `DESeq2`. To do this, we use the `DESeqDataSetFromTximport()` function that creates a DDS from our `txi` object. This `txi` object contains matrices for counts, abundance, and transcript length, and `DESeq2` only expects counts, sample info, and the design formula. 


In [ ]:
dds <- DESeqDataSetFromTximport(
  txi,
  colData = colData,
  design = ~ condition
)

head(dds)

### Filtering for counts less than 10

We filter genes with counts less than 10 out of our dataset to remove genes with low expression that could impact our quantification later on. This is due to the way that pseudo-aligners estimate counts using k-mers fractionally.

In [ ]:
keep <- rowSums(counts(dds)) >= 10
dds <- dds[keep,]

## Variance stabilizing transformation

`vst` stands for Variance Stabilizing Transformation. It is a function in DESeq2 that transforms raw or normalized count data into a form suitable for visualization and downstream analysis, while addressing the inherent properties of RNA-seq counts:

- (estimated in the case of pseudo-alignment) RNA-seq counts are discrete, overdispersed, and their variance depends on the mean (higher counts have higher variance).

- Many standard statistical methods (like PCA, clustering, heatmaps) assume constant variance across observations. Raw counts violate this assumption, so we use normalized length-adjusted counts within our analysis.

`vst` transforms counts so that:

- Variance becomes roughly independent of the mean (homoscedasticity).

- Values are on a continuous, approximately normal scale.

We can use these transformed values to produce a heatmap of sample distances. These heatmaps measure the pairwise distance between samples based on their expression data. Each cell represents the distance between two samples, and samples are usually clustered with dendrograms. This means we can observe distances between samples within each treatment group relative to the sample it is compared to assess the consistency of our treatment groups.

In [ ]:
vsd <- vst(dds, blind=FALSE) #vst function

dist_matrix <- dist(t(assay(vsd)))

pheatmap(as.matrix(dist_matrix), clustering_distance_rows="euclidean", 
         clustering_distance_cols="euclidean", main="Sample distances")

In [ ]:
vst_mat <- assay(vsd)
cor(vst_mat)

## Questions: 
What does the variance stabilizing transformation do to the raw counts in `dds`?

Why might it be problematic to use raw counts directly in dist() without VST or rlog transformation?

What information can you infer from the clustering pattern in the heatmap?

How would the heatmap change if you used "maximum" or "manhattan" instead of "euclidean" as the clustering distance?

In [ ]:
#Do this below, generating a set of new heatmaps with different clustering distances:


## PCA plots

Principal component analysis (PCA) is a dimensionality reduction technique used to explain variance. If you have never seen a PCA plot and are unfamiliar with dimensionality reduction, please watch the video below: 

[Interpretation of Principal Component Analysis (PCA) in RNA seq](https://www.youtube.com/watch?v=aUOV7EBGOy8)

Briefly, PCA plots are used to reduce data with a large number of variables, in our case, thousands of genes, into principal components that capture the pattern of variation within our sample data. Each principal component(PC1) is a new axis within the data, used to show the relation between variables. PC1 shows the largest amount of variation, PC2 explains the second, and so on. We can attribute the amounts of total variance within the data relative to each PC, and also plot our PCs to visualize how the samples relate to one another. 

A PCA plot is a scatter plot where each point represents a sample. The x-axis is PC1, and the y-axis is PC2. Points (samples) that are closer together have similar gene expression profiles, and points that are far apart have different gene expression profiles. We can quickly observe these PCA plots to assess how our samples and treatment groups cluster, allowing us to identify patterns and outliers within the data. 

PCA plots differ from heatmaps as they summarize the overall variation within the data contributed to PC1 and PC2, capturing how samples relate globally based on their gene expression patterns. 

In [ ]:
plotPCA(vsd, intgroup="condition") + ggtitle("PCA of Salmon counts")

## **Question**

Interpret this PCA plot in terms of sample clustering and the variance accounted for by PC1 and PC2.

How much of the sample clustering is attributed to PC1 and PC2?

What does sample clustering tell us about the differences between our two treatment groups?

What biological factors may explain the variation along PC2 within the sample groups, as PC1 seems to separate the treatment conditions? Do you think PC2 relates to an iron-deficiency response? why?

## Differential Expression analysis

Differential expression (DE) analysis is used to understand how gene expression is affected by the treatment condition. In our case, we are measuring the impact of the OPT3 mutation in *A. thaliana*. Using DE analysis, the up- or down-regulation of genes can be calculated using `Log2FoldChange` or log2FC. This value is used to interpret changes in gene expression in a way that scales gene expression counts so that they can be visualized easily. For more information on how `DESeq2` works, watch this video: 

[Log2 fold-change & DESeq2 model in a nutshell](https://www.youtube.com/watch?v=mq6UvDneKc0)

Running `DESeq2` is simple, and we will cover in detail how it fits log2FC by gene expression within Module 4. For now, just understand that `DESeq2` needs the following input: 

dds - `DESeqDataSet`

This includes the following: 
```
dds <- DESeqDataSetFromTximport(
  txi,
  colData = colData,
  design = ~ condition
)
```
We then use the `results()` function to generate results regarding our two treatment conditions (OPT3, Col-0). 

Results tables are generated using the function results, which extracts a results table with log2 fold changes, p-values, and adjusted p-values.

[DESeq2 Documentation](https://bioconductor.org/packages/devel/bioc/vignettes/DESeq2/inst/doc/DESeq2.html)




In [ ]:
dds <- DESeq(dds)
res <- results(dds, contrast=c("condition","OPT3","Col0"))
res <- lfcShrink(dds, coef="condition_OPT3_vs_Col0", type="apeglm") # Shrink log2FC
summary(res)

When we print `res`, our `dds` results, we get a table describing the following: 

- `baseMean` - average normalized expression of a gene across all samples
- `log2FoldChange` - estimated Log2FC in expression between the 2 treatment conditions
- `lfcSE` - Log2FC standard error, with higher values meaning more variability
- `pvalue` - The raw p-value from the hypothesis test of whether the gene is differentially expressed (whether L2FC != 0), without adjustments
- `padj` - p-value calculated using the Benjamini–Hochberg method to control the false discovery rate (FDR). This is the value you generally use to decide statistical significance (e.g., padj < 0.05).

In [ ]:
res

In [ ]:
# let's filter for a list of significant genes. These were different between the Col-0 and OPT3 treatments 
sig_genes <- rownames(res)[which(res$padj < 0.05 & !is.na(res$padj))]

length(sig_genes) # how many significant genes?

Lets look at the VST's for all of our significant (padj < 0.05) genes.

In [ ]:
# Use variance-stabilized counts
vst_counts <- assay(vst(dds))[sig_genes, ]

# Ensure there are finite values
vst_counts <- vst_counts[apply(vst_counts, 1, function(x) any(is.finite(x))), ]

                               
ComplexHeatmap::Heatmap(vst_counts,
                           name = "VST",
                           show_row_names = FALSE,
                           show_column_names = TRUE,
                           cluster_rows = TRUE,
                           cluster_columns = TRUE)

This graph shows us that most genes (y-axis) behave similarly across conditions, and that our samples (x-axis) behave similarly within treatment groups. All is looking good so far, so let's move on to looking at the differences in log2fc.

In [ ]:
sig_genes <- rownames(res)[which(res$padj < 0.05 & !is.na(res$padj))]

sig_genes <- rownames(res)[order(res$padj)][1:25]

mat <- assay(vst(dds))[sig_genes, ]  # e.g., using variance-stabilized counts

if(nrow(mat) > 0 && ncol(mat) > 0){
  ComplexHeatmap::Heatmap(mat, name = "VST",)
} else {
  message("Matrix is empty; skipping heatmap.")
}

## Plotting of our `res` data

Let's now look at our Log2FC between our samples using our cut-off of 1 relative to their adjusted p-value (`padj`). With this volcano plot, we can quickly identify up- and down-regulated genes with high significance. Samples that pass our biologically relevant cut-off of 1 Log2FC are highlighted in red. 

In [ ]:
ggplot(res %>% as.data.frame() %>% rownames_to_column("gene"), 
       aes(x=log2FoldChange, y=-log10(padj))) +
  geom_point(alpha=0.5) +
  geom_point(data=subset(res, padj<0.05 & abs(log2FoldChange)>1), color="red") +
  theme_minimal() +
  ggtitle("Volcano plot: OPT3 vs Col0") +
  xlab("log2 fold change") + ylab("-log10 adjusted p-value")

## Filtering for DEGs within our results from `DESeq2`

We can see in the volcano plot above, we have lots of DEG's that meet our 1 log2FC cut-off. To compare the differential expression of our genes, we need to set a limit to see the change in gene expression across samples. The Log2FC needed can be adjusted based on our experimental conditions. Generally, a Log2FC value of 1 is used. 

In [ ]:
#Subset the results for genes that are significant, and have a log2FC greater than 1
deg <- res %>% 
  as.data.frame() %>%
  rownames_to_column("gene") %>%
  filter(padj < 0.05 & abs(log2FoldChange) > 1)

nrow(deg) #How many DEGS with L2FC > 1?

deg #Print DEG list

## Extracting the top 10 up-regulated and down-regulated DEGs within `deg`

Now, it is your turn to practice some R scripting. Use `deg` to make two data frames called `top_10_deg_up` and `top_10_deg_down`. 

Hint: Make two dataframes from `deg`, filtering by order for the Log2FC and selecting the 10 highest and lowest values for each respective dataframe. 

In [ ]:
# Subset the two data frames top_10_deg_up and top_10_deg_down here: 


## Annotating genes using `biomaRt`

We can annotate our genes using the GeneID so that we do not have to individually search for each of them:

In [ ]:
if (!requireNamespace("biomaRt", quietly = TRUE)) {
  BiocManager::install("biomaRt")
}

#select the 10 most significant genes
deg_genes <- deg$gene[order(deg$padj)]


# Connect to Ensembl Plants and pull A. thaliana dataset
mart <- useMart(
  biomart = "plants_mart",
  dataset = "athaliana_eg_gene",
  host = "https://plants.ensembl.org"
)

#gather gene info
gene_info <- getBM(
  attributes = c("ensembl_gene_id", "external_gene_name"),
  filters = "ensembl_gene_id",
  values = deg,
  mart = mart
)

# Map back to top_genes order
gene_map <- setNames(gene_info$external_gene_name, gene_info$ensembl_gene_id)
gene_names <- gene_map[deg_genes]
gene_names[is.na(gene_names)] <- deg_genes[is.na(gene_names)]  # keep original IDs if no mapping

deg <- deg %>%
  mutate(common_name = gene_map[gene])
deg

## **Question:** 

What are the top 10 up- and down-regulated genes' common names?

What do their adjusted p-values indicate about their significance?

HINT: Now that we have added the `common_name` column to our `deg` dataframe, make the `top_10_deg_up` and `top_10_deg_down` data frames again, and print their geneIDs and their common names. 

In [ ]:
#Your code here: 


## How to interpret our counts? What should we do for a meaningful analysis?

We can observe our normalized counts for each gene relative to treatment (Col-0 or OPT3 mutant). Below, you will find boxplots of the top 20 up- and down-regulated genes normalized counts. While these can be good for reporting or performing our own log2FC calculations, they are not very useful in a data set coming from pseudo-alignment, as the counts are estimated. Observe the count changes for our top 20 DEGs:

In [ ]:
# Now merge the two data sets into one, called top_genes
top_genes <- c(top_10_deg_up$gene, top_10_deg_down$gene)
top_genes

In [ ]:
# Use normalized counts
norm_counts <- counts(dds, normalized = TRUE)

# Loop through top genes only
for (i in seq_along(top_genes)) {
  g <- top_genes[i]              # AGI ID (e.g. AT4G19690)
  gene_name <- gene_map[g]     # Common name from biomart
  
  # Extract normalized counts for this one gene
  df <- data.frame(
    count = norm_counts[g, ],
    condition = colData(dds)$condition
  )
  
  # Plot
  p <- ggplot(df, aes(x = condition, y = count)) +
    geom_boxplot(fill = c("#56B4E9", "#E69F00")) +
    geom_jitter(width = 0.1) +
    labs(
      title = paste("Expression of", gene_name, "(", g, ")"),
      y = "Normalized counts",
      x = "Condition"
    ) +
    theme_minimal()
  
  print(p)
}

A better way to visualize the changes in gene expression within such a small sample set would be by looking at the changes between the VST counts. These counts are stabilized for variance and transformed to avoid heterodasticity (low count genes dominating the differences in our comparison). This means **the VST will show the absolute, variance-stabilized expression of genes across samples**. Look below at the top 20 up- and down-regulated genes' VST counts:

In [ ]:
vsd <- vst(dds)
vst_mat <- assay(vsd)[top_genes, ]
# Map rownames to common names
row_labels <- deg$common_name[match(rownames(vst_mat), deg$gene)]

# If any are NA, fall back to the original IDs
row_labels[is.na(row_labels)] <- rownames(vst_mat)[is.na(row_labels)]

# Plot heatmap of VST's
pheatmap(vst_mat,
         cluster_rows = TRUE,
         cluster_cols = TRUE,
         scale = "none",  # raw VST counts
         labels_row = row_labels,
         main = "VST Counts Heatmap with Gene Names")

We can also observe the z-scores across each of the top 20 up- and down-regulated genes within `top_gene` to highlight each gene's relative deviation from the mean across samples. **Z scores are calculated with this formula:** 


$$Z = \frac{x - \mu}{\sigma}$$

(score $x$, mean $\mu$, standard deviation $\sigma$)

where `x` is the expression of a gene in a sample, `μ` is the mean across samples, and `σ` is the standard deviation.


- Positive Z-score - gene is higher than average in that sample

- Negative Z-score - gene is lower than average in that sample

Z-scores show relative up- or downregulation within each gene, independent of absolute expression magnitude.

In [ ]:
# Z-score normalization (row-wise)
zscore_mat <- t(scale(t(vst_mat)))  # scale each row (gene)

# Heatmap of Z-scores
pheatmap(zscore_mat,
               cluster_rows = TRUE,
               cluster_cols = TRUE,
               scale = "none",  # already scaled
               labels_row = row_labels,
               main = "Z-score Heatmap")


We can use a combination of our VST's and Z scores to interpret our findings about each of our genes in a biological context: 
- A high VST count with a high Z-score - gene is both highly expressed and strongly upregulated in that sample relative to others.

- A low VST count with a low Z-score - gene is lowly expressed and downregulated.

- Genes with moderate VST but extreme Z-scores - smaller absolute changes despite showing strong relative regulation across conditions.

## **Question:** 

Look at FRO5's and CYP82C4's VST score, Z-score, and the normalized counts. What conclusions can you draw from these graphs, and how will they impact the investigation of these genes within our small sample size?

# Investigating iron homeostasis using our DE analysis
From here on, we will investigate iron homeostasis within our example data set as a means to check the validity of our results. **We know from literature that an OPT3 mutation should induce an iron deficiency response within the roots of our *A. thaliana* samples**. We should first check the necessity of this using a functional analysis of **all** of our DEGs. We can do this by using gene-ontology enrichment tools such as `enrichGO`. These packages will use the AGI loci, or gene ID, of our DEGs to compare them to known associations for these genes. Below is an example: 

In [ ]:
if (!requireNamespace("org.At.tair.db", quietly = TRUE)) {
  BiocManager::install("org.At.tair.db")
}
library(org.At.tair.db)

deg_gene_list <- deg$gene

ego <- enrichGO(gene = deg_gene_list,
                OrgDb = org.At.tair.db,
                keyType = "TAIR",
                ont="BP",
                pAdjustMethod="BH",
                qvalueCutoff=0.05)
dotplot(ego) + ggtitle("GO enrichment of DEGs")


Knowing that many of our DEGs from the `DESeq2` analysis are associated with iron homeostasis within our GO search, we can start to investigate some genes that have been identified to be up- or down-regulated in response to iron. First, we will compile a list of known genes affected by iron levels within our *A. thaliana* plants: 

In [ ]:
iron_genes <- c("IRT1", "FRO2", "FIT", "FRD3", "NAS1", "NAS2", "OPT3")

iron_gene_map <- data.frame(
  symbol = c("IRT1", "FRO2", "FIT", "FRD3", "NAS1", "NAS2", "OPT3"),
  agi_id = c("AT4G19690", "AT1G01580", "AT2G28160", "AT3G08040",
             "AT5G04950", "AT5G04960", "AT2G13610")
)

iron_degs <- res[rownames(res) %in% iron_gene_map$agi_id, ]

# Optional: add gene symbols for clarity
iron_degs$symbol <- iron_gene_map$symbol[match(rownames(iron_degs), iron_gene_map$agi_id)]

# View the filtered DEGs
iron_degs

Now, we will check to see if there is a correlation between the iron response genes within `iron_genes` and our 6 samples. Ideally, there will be a strong correlation between the VST counts of the genes in `iron_genes` and our sample treatments (OPT3 and Col0):

In [ ]:
# Subset VST matrix to only iron genes
vst_mat_iron <- assay(vsd)[rownames(vsd) %in% iron_gene_map$agi_id, ]

# Add gene symbols for clarity
rownames(vst_mat_iron) <- iron_gene_map$symbol[match(rownames(vst_mat_iron), iron_gene_map$agi_id)]

# Descriptive stats per gene per group
gene_stats <- data.frame()
for(g in rownames(vst_mat_iron)) {
  for(grp in unique(dds$condition)) {
    vals <- vst_mat_iron[g, dds$condition == grp]
    gene_stats <- rbind(gene_stats, 
                        data.frame(
                          gene = g,
                          group = grp,
                          mean = mean(vals),
                          median = median(vals),
                          sd = sd(vals)
                        ))
  }
}
gene_stats


# Correlation analysis between samples (all iron genes)
cor_mat <- cor(vst_mat_iron)
cor_mat
# visualize with heatmap
library(pheatmap)
pheatmap(cor_mat, main = "Sample-Sample Correlation (Iron Genes)")

There is a strong correlation between the VSTs of `iron_genes` across the 6 samples we have. All of our samples show a high correlation within their treatment groups(OPT3 and Col-0) but not between the two treatment groups.

We are on the right track!

Let's create a heatmap of the VST scores for each gene within `iron_genes`: 

In [ ]:
intersect(rownames(vst_mat), iron_gene_map$agi_id)

In [ ]:
# Subset VST matrix to only iron genes
vst_mat_iron <- assay(vsd)[rownames(vsd) %in% iron_gene_map$agi_id, ]
# Create the heatmap
pheatmap(vst_mat_iron,
         scale = "row", 
         cluster_rows = TRUE, 
         cluster_cols = TRUE,
         labels_row = iron_gene_map$symbol[match(rownames(vst_mat_iron), iron_gene_map$agi_id)],
         main = "Expression of Iron Homeostasis Genes using VST")

Now, we will observe the z-scores for the genes within `iron_genes` for each sample using a heat map. Please create the heat map below: 

**Hint: use the cell above for formatting reference**

In [ ]:
# Subset VST matrix to only iron genes
vst_mat_iron <- assay(vsd)[rownames(vsd) %in% iron_gene_map$agi_id, ]

# Compute Z-scores per gene (row-wise)
zscore_mat <- t(scale(t(vst_mat_iron)))  # scale by row

#YOU CREATE A HEATMAP FROM zscore_mat: 

## **Question:**

What are the results of these z-scores and VST counts for the 7 genes within `iron_genes`? What conclusions can we draw about the impact of OPT3 mutation in *A. thaliana* and how confident should we be in these results? 

How would you approach this analysis differently if you were investigating a mutation whose effects were unknown (IE, we didn't expect to see an iron deficiency response)?

## Why are we not plotting log2FC in a heatmap within this dataset?

In the analysis we completed, we used only 6 samples. There were only two treatment conditions (OPT3 and Col-0), and log2FC represents the difference in expression between the two experimental conditions, calculated after normalization and model fitting. Log2FC represents one value per gene for all samples within each treatment condition, and is used to compare across treatments. We can observe the Log2FC for each gene within our experiment, but we are comparing the changes against the control: 

In [ ]:
# Select the iron genes from your DESeq2 results
iron_l2fc <- res[rownames(res) %in% iron_gene_map$agi_id, "log2FoldChange", drop = FALSE]

# Make a matrix for pheatmap
l2fc_mat <- as.matrix(iron_l2fc)

# Label rows with gene symbols
rownames(l2fc_mat) <- iron_gene_map$symbol[match(rownames(l2fc_mat), iron_gene_map$agi_id)]

# Plot log2FC heatmap
pheatmap(l2fc_mat,
         cluster_rows = TRUE,
         cluster_cols = FALSE,  # only one column if single comparison
         color = colorRampPalette(c("blue","white","red"))(100),
         main = "Log2 Fold Change of Iron Homeostasis Genes")

If we wanted to compare the Log2FC for each sample, we would be doing so using our control as the reference for calculating log2FC, so the results are not desirable...
It is okay to generate heatmaps of log2FC when you are making pairwise comparisons across multiple conditions, but in our case, this is not very helpful. In this experiment, we used log2FC to produce our volcano plots, as well as to identify a cut-off for our DEGs. See below for an example comparing log2FC across the samples within the experiment: 

![image.png](images/M3E1_RNA_Seq_Analysis_f09ce4e8.png)

## Next steps:
In Module 4, we will familiarize ourselves with `R` and practice using and comparing different packages that fit into our NGS pipelines. We will practice DE analysis using a sample set, making comparisons across treatment conditions, and comparing our results to published data to visualize how changes within the analysis directly affect the results. 